In [ ]:
%matplotlib inline
import os, random, math, time
import numpy as np
import cv2
import matplotlib.pyplot as plt
from glob import glob

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils import spectral_norm
from torchvision.utils import make_grid, save_image

# reproducibilidad
manualSeed = 33
random.seed(manualSeed)
np.random.seed(manualSeed)
torch.manual_seed(manualSeed)

FRAMES_ROOT = "/kaggle/input/proyecto-final/frames"  
SAMPLE_DIR = "samples"
os.makedirs(SAMPLE_DIR, exist_ok=True)

# Device / GPUs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()
print("Device:", device, "GPUs:", n_gpus)
if device.type == "cuda":
    for i in range(n_gpus):
        print("  GPU", i, ":", torch.cuda.get_device_name(i))

IMAGE_SIZE = 128          
BATCH_SIZE_PER_GPU = 32  
BATCH_SIZE = max(1, BATCH_SIZE_PER_GPU * max(1, n_gpus))
EPOCHS = 100
LR = 2e-4
BETA1 = 0.5
LAMBDA_L1 = 100.0         
SAMPLE_INTERVAL = 1      
NUM_WORKERS = 2
PRINT_FREQ = 20

In [ ]:
def resize_and_pad(img, size=(IMAGE_SIZE, IMAGE_SIZE), pad_value=127):
    """Redimensiona manteniendo aspect ratio y centros con padding"""
    target_h, target_w = size
    h, w = img.shape[:2]
    scale = min(target_w / w, target_h / h)
    nw, nh = max(1, int(w * scale)), max(1, int(h * scale))
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    pad_w = target_w - nw
    pad_h = target_h - nh
    top = pad_h // 2
    bottom = pad_h - top
    left = pad_w // 2
    right = pad_w - left
    padded = cv2.copyMakeBorder(resized, top, bottom, left, right,
                                cv2.BORDER_CONSTANT, value=[pad_value, pad_value, pad_value])
    return padded

def load_frames_from_folder(folder):
    files = sorted([os.path.join(folder, f) for f in os.listdir(folder)])
    frames = []
    for i, f in enumerate(files):
        img = cv2.imread(f)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        frames.append(img)
    return frames

def make_triplets_from_videos(frames_list, target_size=(IMAGE_SIZE, IMAGE_SIZE)):
    A_list, B_list, MID_list = [], [], []
    for vid_idx, frames in enumerate(frames_list):
        N = len(frames)
        if N < 3:
            continue
        for i in range(N - 2):
            a = resize_and_pad(frames[i], target_size)
            mid = resize_and_pad(frames[i+1], target_size)
            b = resize_and_pad(frames[i+2], target_size)
            A_list.append(a); MID_list.append(mid); B_list.append(b)
        print(f"Video {vid_idx}: generadas {N-2} tripletas")
    A = np.array(A_list, dtype=np.float32)
    B = np.array(B_list, dtype=np.float32)
    MID = np.array(MID_list, dtype=np.float32)
    # Normalizar a [-1,1] y permutar a (N,C,H,W)
    def norm(x):
        x = (x - 127.5) / 127.5
        return x.transpose(0,3,1,2)
    return norm(A), norm(B), norm(MID)

folders = [os.path.join(FRAMES_ROOT, f"video{i}") for i in range(3)]
frames_all = []
for i, folder in enumerate(folders):
    print("Cargando:", folder)
    frames = load_frames_from_folder(folder)
    frames_all.append(frames)

A_np, B_np, MID_np = make_triplets_from_videos(frames_all, target_size=(IMAGE_SIZE, IMAGE_SIZE))
print("Tripletas:", A_np.shape, B_np.shape, MID_np.shape)

X_inputs = np.concatenate([A_np, B_np], axis=1)  # (N,6,H,W)
dataset = TensorDataset(torch.from_numpy(X_inputs), torch.from_numpy(MID_np))
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
print("Batches por época:", len(dataloader))

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1 or classname.find('Linear') != -1:
        try:
            nn.init.normal_(m.weight.data, 0.0, 0.02)
        except Exception:
            pass
    elif classname.find('BatchNorm') != -1:
        try:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
            nn.init.constant_(m.bias.data, 0)
        except Exception:
            pass

def conv_block(in_c, out_c, k=4, s=2, p=1, act=True):
    layers = [spectral_norm(nn.Conv2d(in_c, out_c, k, s, p, bias=False)),
              nn.BatchNorm2d(out_c)]
    if act:
        layers.append(nn.LeakyReLU(0.2, inplace=True))
    return nn.Sequential(*layers)

def up_block(in_c, out_c):
    return nn.Sequential(
        nn.Upsample(scale_factor=2, mode='nearest'),
        spectral_norm(nn.Conv2d(in_c, out_c, 3, 1, 1, bias=False)),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True)
    )

class CondGeneratorUNet(nn.Module):
    def __init__(self, in_channels=6, out_channels=3, ngf=32):
        super().__init__()
        self.e1 = conv_block(in_channels, ngf, k=4, s=2, p=1)
        self.e2 = conv_block(ngf, ngf*2)
        self.e3 = conv_block(ngf*2, ngf*4)
        self.e4 = conv_block(ngf*4, ngf*8)
        self.b = nn.Sequential(
            spectral_norm(nn.Conv2d(ngf*8, ngf*8, 3, 1, 1, bias=False)),
            nn.BatchNorm2d(ngf*8),
            nn.ReLU(inplace=True)
        )
        self.u4 = up_block(ngf*8, ngf*4)
        self.u3 = up_block(ngf*8, ngf*2)
        self.u2 = up_block(ngf*4, ngf)
        self.u1 = up_block(ngf*2, ngf)
        self.final = nn.Sequential(spectral_norm(nn.Conv2d(ngf, out_channels, 3, 1, 1, bias=False)),
                                   nn.Tanh())
        self.apply(weights_init)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(e1)
        e3 = self.e3(e2)
        e4 = self.e4(e3)
        b  = self.b(e4)
        u4 = self.u4(b)
        u4 = torch.cat([u4, e3], dim=1)
        u3 = self.u3(u4)
        u3 = torch.cat([u3, e2], dim=1)
        u2 = self.u2(u3)
        u2 = torch.cat([u2, e1], dim=1)
        u1 = self.u1(u2)
        out = self.final(u1)
        return out


class CondDiscriminator(nn.Module):
    def __init__(self, in_channels=9, ndf=32):
        super().__init__()
        self.net = nn.Sequential(
            spectral_norm(nn.Conv2d(in_channels, ndf, 4, 2, 1, bias=False)), nn.LeakyReLU(0.2, True),
            spectral_norm(nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False)), nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, True),
            spectral_norm(nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False)), nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, True),
            spectral_norm(nn.Conv2d(ndf*4, ndf*8, 4, 1, 1, bias=False)), nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, True),
            spectral_norm(nn.Conv2d(ndf*8, 1, 4, 1, 0, bias=False))   # patch logits
        )
        self.apply(weights_init)

    def forward(self, cond_AB, mid):
        x = torch.cat([cond_AB, mid], dim=1)  # (B,9,H,W)
        return self.net(x)  # logits (B,1,h',w')

In [ ]:
G = CondGeneratorUNet(in_channels=6, out_channels=3, ngf=32).to(device)
D = CondDiscriminator(in_channels=9, ndf=32).to(device)

if n_gpus > 1:
    G = nn.DataParallel(G)
    D = nn.DataParallel(D)

optimizerG = optim.Adam(G.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizerD = optim.Adam(D.parameters(), lr=LR, betas=(BETA1, 0.999))

adversarial_criterion = nn.BCEWithLogitsLoss().to(device)
l1_criterion = nn.L1Loss().to(device)

real_label = 0.9
fake_label = 0.0

import torch.amp
scalerG = torch.amp.GradScaler()
scalerD = torch.amp.GradScaler()

fixed_cond = None
fixed_mid = None

def detach_cpu(t):
    return t.detach().cpu()

def sample_and_show(cond_AB_batch, real_mid_batch, epoch, n_show=4, show=True):
    G.eval()
    with torch.no_grad():
        cond = cond_AB_batch[:n_show].to(device)
        real_mid = real_mid_batch[:n_show].to(device)
        fake_mid = G(cond)
        # map [-1,1] -> [0,1]
        cond0 = (cond[:, :3] + 1) / 2.0
        cond1 = (cond[:, 3:] + 1) / 2.0
        fake = (fake_mid + 1) / 2.0
        real = (real_mid + 1) / 2.0

        rows = []
        for i in range(n_show):
            # concat A | fake | real | B horizontally
            row = torch.cat([cond0[i], fake[i], real[i], cond1[i]], dim=2)  # cat on width
            rows.append(row)
        grid = torch.stack(rows, dim=0)
        grid_img = make_grid(grid, nrow=1)  # each row is one sample
        out_path = os.path.join(SAMPLE_DIR, f'epoch_{epoch:04d}.png')
        save_image(grid_img, out_path)
        if show:
            np_img = grid_img.cpu().permute(1,2,0).numpy()
            plt.figure(figsize=(12, 3*n_show))
            plt.imshow(np_img)
            plt.axis('off')
            plt.title(f'Epoch {epoch}')
            plt.show()
    G.train()



# Training loop

In [ ]:
G_losses = []
D_losses = []
img_list = []   

print("Start training...")
for epoch in range(EPOCHS):
    epoch_g_losses = []
    epoch_d_losses = []
    if fixed_cond is None:
        # pick first batch as fixed examples for monitoring
        it = iter(dataloader)
        fixed_cond, fixed_mid = next(it)
        fixed_cond = fixed_cond.to(device); fixed_mid = fixed_mid.to(device)

    for i, (cond_AB, real_mid) in enumerate(dataloader):
        cond_AB = cond_AB.to(device, non_blocking=True)
        real_mid = real_mid.to(device, non_blocking=True)

        b_size = cond_AB.size(0)
        # --------------- Train D ---------------
        optimizerD.zero_grad()
        with torch.cuda.amp.autocast():
            fake_mid = G(cond_AB)
            # D outputs logits map
            real_logits = D(cond_AB, real_mid)   # (B,1,h,w)
            fake_logits = D(cond_AB, fake_mid.detach())

            # flatten per-sample: mean the logits map to a scalar (works for BCE)
            real_scores = real_logits.view(b_size, -1).mean(dim=1)
            fake_scores = fake_logits.view(b_size, -1).mean(dim=1)

            real_labels = torch.full_like(real_scores, real_label, device=device)
            fake_labels = torch.full_like(fake_scores, fake_label, device=device)

            lossD_real = adversarial_criterion(real_scores, real_labels)
            lossD_fake = adversarial_criterion(fake_scores, fake_labels)
            lossD = (lossD_real + lossD_fake) * 0.5

        scalerD.scale(lossD).backward()
        scalerD.step(optimizerD)
        scalerD.update()

        # --------------- Train G ---------------
        optimizerG.zero_grad()
        with torch.cuda.amp.autocast():
            fake_mid = G(cond_AB)
            fake_logits_for_g = D(cond_AB, fake_mid)
            fake_scores_g = fake_logits_for_g.view(b_size, -1).mean(dim=1)
            # adversarial loss (want D to predict real_label)
            target_for_g = torch.full_like(fake_scores_g, real_label, device=device)
            adv_loss = adversarial_criterion(fake_scores_g, target_for_g)
            l1_loss = l1_criterion(fake_mid, real_mid)
            lossG = adv_loss + LAMBDA_L1 * l1_loss

        scalerG.scale(lossG).backward()
        scalerG.step(optimizerG)
        scalerG.update()

        epoch_d_losses.append(lossD.item())
        epoch_g_losses.append(lossG.item())

        if (i % PRINT_FREQ) == 0:
            print(f"Epoch [{epoch}/{EPOCHS}] Batch [{i}/{len(dataloader)}]  D_loss: {lossD.item():.4f}  G_loss: {lossG.item():.4f}  L1: {l1_loss.item():.4f}")

    avg_d = float(np.mean(epoch_d_losses))
    avg_g = float(np.mean(epoch_g_losses))
    D_losses.append(avg_d); G_losses.append(avg_g)
    print(f"==> Época {epoch}/{EPOCHS} | D_loss: {avg_d:.4f} | G_loss: {avg_g:.4f}")

    # guardar + mostrar cada sample_interval épocas
    if (epoch % SAMPLE_INTERVAL) == 0:
        sample_and_show(fixed_cond, fixed_mid, epoch, n_show=min(4, fixed_cond.size(0)), show=True)
        # store for animation if wanted
        img = make_grid(( (G(fixed_cond[:4].to(device)).detach()+1)/2.0 ), nrow=4, normalize=False)
        img_list.append(img.cpu())

# Plots

In [ ]:
plt.figure(figsize=(10,5))
plt.title("Generator and Discriminator Loss During Training (per epoch)")
plt.plot(G_losses, label="G")
plt.plot(D_losses, label="D")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()

# Final comparison (real vs fake)
real_batch = next(iter(dataloader))[1]  # real_mid batch
with torch.no_grad():
    cond_batch = next(iter(dataloader))[0]  # cond_AB
    fake_batch = G(cond_batch.to(device)[:16]).cpu()

plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.axis("off")
plt.title("Real mids")
plt.imshow(np.transpose(make_grid((real_batch[:16]+1)/2.0, nrow=4).numpy(), (1,2,0)))
plt.subplot(1,2,2)
plt.axis("off")
plt.title("Fake mids (last epoch)")
plt.imshow(np.transpose(make_grid((fake_batch+1)/2.0, nrow=4).numpy(), (1,2,0)))
plt.show()

# (Opcional) crear animación a partir de img_list
try:
    import matplotlib.animation as animationh
    from IPython.display import HTML
    fig = plt.figure(figsize=(6,6))
    ims = []
    for im in img_list:
        imshow_obj = plt.imshow(np.transpose(im.numpy(), (1,2,0)), animated=True)
        plt.axis('off')
        ims.append([imshow_obj])
    ani = animation.ArtistAnimation(fig, ims, interval=800, repeat_delay=1000, blit=True)
    display(HTML(ani.to_jshtml()))
except Exception as e:
    print("No se pudo crear animación (ok):", e)

print("Entrenamiento finalizado. Imágenes guardadas en:", SAMPLE_DIR)

In [14]:
import imageio
from pathlib import Path
gif_path = Path("samples/training_animation.gif")

frames = []
for im in img_list:  # im es un torch.Tensor (C,H,W) en CPU
    np_img = np.transpose(im.numpy(), (1,2,0))  # H,W,C
    # imageio expects uint8 0-255
    np_img = (np_img * 255).astype(np.uint8)
    frames.append(np_img)

imageio.mimsave(str(gif_path), frames, fps=2)   # ajusta fps
print("GIF guardado en:", gif_path)


GIF guardado en: samples/training_animation.gif
